In [ ]:
pip install beautifulsoup4 tqdm undetected-chromedriver selenium nltk

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47214 sha256=90e94f597f593071d8c4cbd285cf4087685dd6606b203f2b9809c572b2484324
  Stored in directory: c:\users\sluca\appdata\local\pip\cache\wheels\cf\a1\db\e1275b6f7259aacd6b045f8bfcb1fcbc93827a3916ba55d5b7
Successfully built undetected-chromedriver
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\sluca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: C:\Users\sluca\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import time
import json
import random
import re
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options
import pandas as pd
import uuid
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sluca\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [ ]:
class AllSidesScraper:
    def __init__(self, output_file='allsides_triplets.jsonl', max_stories=100):
        self.output_file = output_file
        self.base_url = 'https://www.allsides.com'
        self.max_stories = max_stories
        chrome_options = Options()
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--disable-gpu")
        self.driver = uc.Chrome(options=chrome_options, version_main=148)

    def _smart_delay(self, min_s=2.0, max_s=4.0):
        time.sleep(random.uniform(min_s, max_s))

    def get_external_url(self, allsides_news_url):
        self.driver.get(allsides_news_url)
        self._smart_delay()
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        button = soup.find('a', id=lambda x: x and str(x).startswith('Read-Full-Story'))
        if not button:
            button = soup.find('a', string=lambda text: text and 'Read Full Story' in text)
        return button['href'] if button and button.get('href') else None

    def extract_article_text(self, url):
        self.driver.get(url)
        self._smart_delay()
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        paragraphs = soup.find_all('p')
        text_blocks = [p.get_text(separator=' ', strip=True) for p in paragraphs if len(p.get_text(strip=True)) > 40]
        return ' '.join(text_blocks) if len(text_blocks) > 0 else None

    def get_triplet_blocks(self, current_blocks, seen_topics):
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        containers = soup.find_all('div', class_='flex flex-col flex-wrap col-span-3 gap-0 p-0 mt-0 w-full smmd:mt-2 md:w-3/4 float-end md:pl-4 smmd:gap-4 smmd:flex-row')
        
        for container in containers:
            if len(current_blocks) >= self.max_stories: break
            
            parent = container.find_parent()
            title_elem = parent.find(['h2', 'h3'], class_=re.compile(r'text-lg', re.I))
            topic_title = title_elem.get_text(strip=True) if title_elem else "Unknown Topic"
            
            if topic_title in seen_topics: continue
            
            links = {}
            for bias in ['left', 'center', 'right']:
                news_item = container.find('div', class_=f'news-item flex-1 {bias}')
                if news_item:
                    a_tag = news_item.find('a', href=True)
                    source_tag = news_item.find('p', class_='news-source-title')
                    if a_tag:
                        links[bias] = {
                            'internal_url': urljoin(self.base_url, a_tag['href']),
                            'source_name': source_tag.get_text(strip=True) if source_tag else "Unknown"
                        }
            
            if 'left' in links and 'center' in links and 'right' in links:
                current_blocks.append({'topic': topic_title, 'links': links})
                seen_topics.add(topic_title)
        return current_blocks

    def run(self, start_url):
        seen_topics = set()
        try:
            with open(self.output_file, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip(): seen_topics.add(json.loads(line).get('topic'))
        except FileNotFoundError: pass

        self.driver.get(start_url)
        all_blocks = self.get_triplet_blocks([], seen_topics)
        
        valid_count = 0
        for block in tqdm(all_blocks, desc="Scraping"):
            try:
                triplet = {'topic': block['topic'], 'sources': {}}
                success = True
                for bias in ['left', 'center', 'right']:
                    ext = self.get_external_url(block['links'][bias]['internal_url'])
                    text = self.extract_article_text(ext) if ext else None
                    if text:
                        triplet[bias] = text
                        triplet['sources'][bias] = block['links'][bias]['source_name']
                    else: success = False; break
                
                if success:
                    with open(self.output_file, 'a', encoding='utf-8') as f:
                        f.write(json.dumps(triplet, ensure_ascii=False) + '\n')
                    valid_count += 1
            except Exception: continue
        
        self.driver.quit()
        print(f"Sessione terminata: {valid_count} nuove triplette salvate.")

In [ ]:
scraper = AllSidesScraper()
scraper.run(start_url='https://www.allsides.com/recent-headline-roundups')

In [17]:
scraper = AllSidesScraper()
scraper.run(start_url='https://www.allsides.com/recent-headline-roundups?page=2')

Scraping:   0%|          | 0/42 [00:00<?, ?it/s]

Sessione terminata: 39 nuove triplette salvate.


In [15]:
def process_triplets_to_dataframe(jsonl_file_path, num_lead_sentences=3):
    rows=[]

    #Reading the json file
    with open(jsonl_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            triplet = json.loads(line)

            #Extracting the topic
            topic_id=triplet.get('topic', triplet.get('story_url', 'unknown_topic'))

            #Iterating on the 3 political labels
            for orientation in ['left', 'center','right']:
                if orientation in triplet and triplet[orientation]:

                    #Binarization of the lables in 1(left/right) and 0(center)
                    binary_label = 1 if orientation in ['left','right'] else 0
                    full_text = triplet[orientation]

                    #Extracting suorce's name if present
                    source_name = 'Unknown'
                    if ' sources' in triplet and orientation in triplet['sources']:
                        source_name = triplet['sources'][orientation]

                    #Tokenization of sentences
                    sentences = sent_tokenize(full_text)
                    lead_sentences = sentences[:num_lead_sentences]

                    article_row = {
                        'article_id': str(uuid.uuid4()),
                        'topic_id': topic_id,
                        'source_name': source_name,
                        'date': None, 
                        'original_orientation': orientation,
                        'binary_label': binary_label,
                        'full_text': full_text,
                        'lead_sentences': lead_sentences
                    }
                    rows.append(article_row)
                    
    # Creazione finale del DataFrame
    df = pd.DataFrame(rows)
    return df

# Esecuzione
file_input = 'allsides_triplets.jsonl'
df_dataset = process_triplets_to_dataframe(file_input, num_lead_sentences=5)


In [16]:
df_dataset.to_csv('dataset_strutturato_allsides.csv', index=False, encoding='utf-8')